In [2]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader


import pandas as pd
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view

import matplotlib.pyplot as plt



In [44]:
df = pd.read_csv("AAPL.csv")
close = np.array(df["Close"])
returns = close[1:]/close[:-1]
log_returns = np.log(returns)

rvolwin  = 10
windows = sliding_window_view(log_returns, window_shape=rvolwin)
rvol = windows.std(axis=1, ddof=1)

log_returns = log_returns[rvolwin-1:]
combined = np.vstack((rvol, 
              log_returns))

rvol_sd = np.std(rvol)
rvol_mean = np.mean(rvol)

log_returns_sd = np.std(log_returns)
log_returns_mean = np.mean(log_returns)

combined[0,:] =  (combined[0,:] - rvol_mean)/rvol_sd
combined[1,:] =  (combined[1,:] - log_returns_mean)/log_returns_sd

seq_len = 5

X = []
Y = []

for i in range(combined.shape[1] - seq_len - (rvolwin - 1) - 1):
    X.append( combined[:, i : i+seq_len] )           
    idx = i + seq_len      
    Y.append( combined[0, idx] )

X = torch.tensor(X).float()
Y = torch.tensor(Y).float()

X = X.reshape((X.shape[0],
               X.shape[2],
               X.shape[1]))
          

test_size = 1000

fh = seq_len + rvolwin - 1

train_X = X[:-test_size - fh]
train_Y = Y[:-test_size - fh]
test_X  = X[-test_size - fh:-fh]
test_Y  = Y[-test_size-fh:-fh]


class model(nn.Module):
    def __init__(self, input_size=2, hidden_size=100, output_size=1):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.rnn(x)
        out = self.fc(out[:, -1, :]) 
        return out

train_dataset = TensorDataset(train_X, train_Y.unsqueeze(1))
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)

test_dataset = TensorDataset(test_X, test_Y.unsqueeze(1))
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

model = model()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr= 0.0001)

num_epochs = 5
for epoch in range(num_epochs):
    model.train()
    for batch_x, batch_y in train_loader:
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss.item():.4f}")


model.eval()
test_loss = 0
with torch.no_grad():
    for batch_x, batch_y in test_loader:
        outputs = model(batch_x)
        test_loss += criterion(outputs, batch_y).item()
test_loss /= len(test_loader)
print(f"Test Loss: {test_loss:.4f}")

Epoch 1/5, Loss: 0.0933
Epoch 2/5, Loss: 0.0755
Epoch 3/5, Loss: 0.0646
Epoch 4/5, Loss: 0.0615
Epoch 5/5, Loss: 0.0603
Test Loss: 0.0227


In [45]:
pred_vol = model(test_X).detach().squeeze() * rvol_sd + rvol_mean  
actual = test_Y * rvol_sd + rvol_mean

print(pred_vol[:9])
print(actual[:9])


tensor([0.0276, 0.0316, 0.0329, 0.0326, 0.0334, 0.0334, 0.0304, 0.0287, 0.0195])
tensor([0.0316, 0.0335, 0.0334, 0.0342, 0.0339, 0.0307, 0.0290, 0.0197, 0.0194])
